In [2]:
import torch
import torch.nn as nn
import math

from triton.ops import attention

In [3]:
old_repr = torch.Tensor.__repr__
def tensor_info(tensor):
    return repr(tensor.shape)[6:] + ' ' + repr(tensor.dtype)[6:] + '@' + str(tensor.device) + '\n' + old_repr(tensor)
torch.Tensor.__repr__ = tensor_info

print(torch.__version__, torch.cuda.is_available())

2.4.1 True


# AdditiveAttention

In [4]:
query_size,key_size = 1, 1
# parameters
W_q = nn.Linear(query_size,5)
W_k = nn.Linear(key_size,5)
W_v = nn.Linear(5,1)

# query [batch_size,num_queries,h]
q = torch.ones((10,4,5), requires_grad=False)
# key [batch_size,num_keys,h]
k = torch.ones((10,6,5), requires_grad=False)
print(q.shape, k.shape)
# 使用加法广播机制
q = q.unsqueeze(1)
k = k.unsqueeze(2)
f = q + k
f = torch.tanh(f)
print(q.shape,k.shape,f.shape)

score = W_v(f)
print(score.shape)
score = score.squeeze(-1)
print(score.shape)

torch.Size([10, 4, 5]) torch.Size([10, 6, 5])
torch.Size([10, 1, 4, 5]) torch.Size([10, 6, 1, 5]) torch.Size([10, 6, 4, 5])
torch.Size([10, 6, 4, 1])
torch.Size([10, 6, 4])


# DotProductAttention

In [4]:
def masked_softmax(X, valid_lens):  #@save
    def _sequence_mask(X, valid_len, value=0):
        maxlen = X.size(1)
        mask = torch.arange((maxlen), dtype=torch.float32,
                            device=X.device)[None, :] < valid_len[:, None]
        X[~mask] = value
        return X

    if valid_lens is None:
        return nn.functional.softmax(X, dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens, shape[1])
        else:
            valid_lens = valid_lens.reshape(-1)
        # On the last axis, replace masked elements with a very large negative
        # value, whose exponentiation outputs 0
        X = _sequence_mask(X.reshape(-1, shape[-1]), valid_lens, value=-1e6)
        return nn.functional.softmax(X.reshape(shape), dim=-1)

class DotProductAttention(nn.Module):  #@save
    """Scaled dot product attention."""
    def __init__(self, dropout):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    # Shape of queries: (batch_size, no. of queries, d)
    # Shape of keys: (batch_size, no. of key-value pairs, d)
    # Shape of values: (batch_size, no. of key-value pairs, value dimension)
    # Shape of valid_lens: (batch_size,) or (batch_size, no. of queries)
    def forward(self, queries, keys, values, valid_lens=None):
        d = queries.shape[-1]
        # Swap the last two dimensions of keys with keys.transpose(1, 2)
        scores = torch.bmm(queries, keys.transpose(1, 2)) / math.sqrt(d)
        self.attention_weights = masked_softmax(scores, valid_lens)
        return torch.bmm(self.dropout(self.attention_weights), values),self.attention_weights
    
q = torch.ones((10,4,5), requires_grad=False)
k = torch.ones((10,6,5), requires_grad=False)
v = torch.ones((10,6,5), requires_grad=False)
dot_product = DotProductAttention(0)
_, attention_weights = dot_product(q, k, v)

print(attention_weights.shape)

torch.Size([10, 4, 6])


# torch 加法广播机制
翻译过来就是,从两个数组地末尾开始算起,若轴长相等或者其中一个地维度为1,则认为是广播兼容的,否则是不兼容的；
广播兼容的数组会在缺失的维度和长度为1的维度上进行
广播机制牢记一点就可以了:从末尾开始连续找第一个不想同的维度容量开始扩充,直到扩充成维度相同；
例:
图中的第二行:[4, 3]+[1, 3] 末尾3相等, 前移一位, 4!=1
则把1扩充为 4 结束；
图中的第三行:[4, 1]+[1, 3] 末尾 1!=3 将1扩充为3,
迁移一位, 4!=1 把1扩充为4 结束
```python
x = torch.ones((4,1), requires_grad=False)
y = torch.zeros((1,4), requires_grad=False)
print(x, y)
print(x.shape, y.shape)
z = x + y
print(z.shape)
print(z)
```

# 位置编码

In [5]:
class PositionalEncoding(nn.Module):
    """位置编码"""
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        # 创建一个足够长的P
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(
            -1, 1) / torch.pow(10000, torch.arange(
            0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        print("x shape: ", X.shape)
        print(X[:10,:5])
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X)

    def forward(self, X):
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)

In [15]:
pos_encoder = PositionalEncoding(10, 0)
x = torch.zeros((1, 10, 10))
x = pos_encoder(x)

x shape:  torch.Size([1000, 5])
tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.5849e-01, 2.5119e-02, 3.9811e-03, 6.3096e-04],
        [2.0000e+00, 3.1698e-01, 5.0238e-02, 7.9621e-03, 1.2619e-03],
        [3.0000e+00, 4.7547e-01, 7.5357e-02, 1.1943e-02, 1.8929e-03],
        [4.0000e+00, 6.3396e-01, 1.0048e-01, 1.5924e-02, 2.5238e-03],
        [5.0000e+00, 7.9245e-01, 1.2559e-01, 1.9905e-02, 3.1548e-03],
        [6.0000e+00, 9.5094e-01, 1.5071e-01, 2.3886e-02, 3.7857e-03],
        [7.0000e+00, 1.1094e+00, 1.7583e-01, 2.7867e-02, 4.4167e-03],
        [8.0000e+00, 1.2679e+00, 2.0095e-01, 3.1849e-02, 5.0477e-03],
        [9.0000e+00, 1.4264e+00, 2.2607e-01, 3.5830e-02, 5.6786e-03]])


In [16]:
X = torch.arange(100, dtype=torch.float32).reshape(
            -1, 1) / torch.pow(10000, torch.arange(
            0, 10, 2, dtype=torch.float32) / 10)

# 多头注意力


In [11]:
q = torch.ones((10,4,5), requires_grad=True)
k = torch.ones((10,6,5), requires_grad=True)
v = torch.ones((10,6,5), requires_grad=True)

num_heads, hidden_dim = 4, 5
W_q = nn.Linear(5, num_heads * hidden_dim)
W_k = nn.Linear(5, num_heads * hidden_dim)
W_v = nn.Linear(5, num_heads * hidden_dim)
W_o = nn.Linear(num_heads * hidden_dim, 5)

q_h = W_q(q)
print(f"q_h shape:{q_h.shape}")
k_h = W_k(k)
print(f"k_h shape:{k_h.shape}")
v_h = W_v(v)
print(f"v_h shape:{v_h.shape}")
# 计算DotProductAttention时，q，k，v的最后一维应为hidden_dim,前面的代码中为了便于计算，将qkv矩阵输出结果在最后一维进行了拼接，因此需要将其还原回[-1,num_head,num_queries,hidden_dim]
q_h = q_h.reshape(-1, 4, num_heads, hidden_dim) # 先切分最后一维
q_h = q_h.permute(0, 2, 1, 3) # 将其转换为[batch_size, num_heads, num_queries, hidden_dim]
q_h = q_h.reshape(-1, 4, hidden_dim) 
# 转换为[batch_size*num_heads, num_queries, hidden_dim], 
# 因为先前实现的DotProductAttention都输入为[batch_size, num_queries, hidden_dim].

# 同样的方法处理 k，v，代码省略
k_h = torch.ones((40, 6, 5))
v_h = torch.ones_like(k_h)


attention = DotProductAttention(0)
output_h, att_weights = attention(q_h, k_h, v_h, valid_lens=None)
# output_h 的shape为 [batch_size*num_heads, num_queries, hidden_dim], 
# 但是 W_o 需要的输入为 [batch_size, num_queries, hidden_dim * num_heads], 
# 因此对output进行逆向处理

output_h = output_h.reshape(-1, num_heads, 4, hidden_dim) #[batch_size, num_heads, num_queries, hidden_dim]
output_h = output_h.permute(0, 2, 1, 3) #[batch_size, num_queries, num_heads, hidden_dim]
output_h = output_h.reshape(-1, 4, num_heads * hidden_dim) # [batch_size, num_queries, num_heads*hidden_dim]

output = W_o(output_h)

print(output.shape)

q_h shape:torch.Size([10, 4, 20])
k_h shape:torch.Size([10, 6, 20])
v_h shape:torch.Size([10, 6, 20])
torch.Size([10, 4, 5])


In [6]:
def transpose_qkv(X, num_heads):
    """为了多注意力头的并行计算而变换形状"""
    # 输入X的形状:(batch_size，查询或者“键－值”对的个数，num_hiddens)
    # 输出X的形状:(batch_size，查询或者“键－值”对的个数，num_heads，num_hiddens/num_heads)
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)

    # 输出X的形状:(batch_size，num_heads，查询或者“键－值”对的个数, num_hiddens/num_heads)
    X = X.permute(0, 2, 1, 3)

    # 最终输出的形状:(batch_size*num_heads,查询或者“键－值”对的个数, num_hiddens/num_heads)
    return X.reshape(-1, X.shape[2], X.shape[3])

def transpose_output(X, num_heads):
    """逆转transpose_qkv函数的操作"""
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, key_size, query_size, value_size, num_hiddens,
                 num_heads, dropout, bias=False, **kwargs):
        super(MultiHeadAttention, self).__init__(**kwargs)
        self.num_heads = num_heads
        self.attention= DotProductAttention(dropout)
        self.W_q = nn.Linear(query_size, num_hiddens*num_heads, bias=bias)
        self.W_k = nn.Linear(key_size, num_hiddens*num_heads, bias=bias)
        self.W_v = nn.Linear(value_size, num_hiddens*num_heads, bias=bias)
        self.W_o = nn.Linear(num_hiddens*num_heads, num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens=None):
        queries = transpose_qkv(self.W_q(queries), self.num_heads)
        keys = transpose_qkv(self.W_k(keys), self.num_heads)
        values = transpose_qkv(self.W_v(values), self.num_heads)

        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)
        output, attention_weight = self.attention(queries, keys, values, valid_lens)
        output_concat = transpose_output(output, self.num_heads)
        return self.W_o(output_concat)

# Transformer
transformer 训练过程中使用了并行化训练，即 \
输入(<_begin_>)，此时目标输出为(<_begin_>,w1) \
输入(<_begin_>,w1)，此时目标输出为(<_begin_>,w1,w2) \
输入(<_begin_>,w1,w2)，此时目标输出为(<_begin_>,w1,w2,w3) \
...\
输入(<_begin_>,w1,..., last_word)，此时目标输出为(<_begin_>,w1,...,last_word,<_end_>)
上述输入可以并行，并以此来计算损失函数。

transformer 预测过程中,trg_input变化如下 \
第一次调用decoder，首先输入(<_begin_>)， mask为[1,0,0,0,...]，预测第一个词；\
第二次调用decoder，首先输入(<_begin_>,w1) ， mask为[1,1,0,0,...]，预测第二个词；\
第三次调用decoder，首先输入(<_begin_>,w1,w2)， mask为[1,1,1,0,...]，预测第三个词；\
...\
直到输出(<_begin_>,w1,w2,...,<_end_>)。\


In [8]:
# transformer
# FFN
class PositionWiseFFN(nn.Module):
    def __init__(self, ffn_num_input, ffn_num_hiddens, ffn_num_outputs,
                 **kwargs):
        super(PositionWiseFFN, self).__init__(**kwargs)
        self.dense1 = nn.Linear(ffn_num_input, ffn_num_hiddens)
        self.relu = nn.ReLU()
        self.dense2 = nn.Linear(ffn_num_hiddens, ffn_num_outputs)

    def forward(self, X):
        return self.dense2(self.relu(self.dense1(X)))
#Add & norm
class AddNorm(nn.Module):
    def __init__(self, normalized_shape, dropout, **kwargs):
        super(AddNorm, self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)
        self.ln = nn.LayerNorm(normalized_shape)
    def forward(self, X, Y):
        return self.ln(self.dropout(Y) + X)


In [ ]:
# encoder layer
class EncoderLayer(nn.Module):
    def __init__(self, key_size, value_size, num_hiddens, num_heads, ffn_num_input, ffn_num_hiddens, ffn_num_outputs, normalized_shape, dropout, bias, **kwargs):
        super(EncoderLayer, self).__init__(**kwargs)
        self.self_attn = MultiHeadAttention(key_size, query_size, value_size, num_hiddens, num_heads, dropout, bias=bias)
        self.add_norm1 = AddNorm(normalized_shape, dropout, bias=bias)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, ffn_num_outputs, **kwargs)
        self.add_norm2 = AddNorm(normalized_shape, dropout, bias=bias)
    
    def forward(self, X, valid_lens):
        Y = self.add_norm1(self.self_attn(X, X, X, valid_lens=valid_lens), X)
        return self.add_norm2(self.ffn(Y), Y)
    
# decoder layer
class DecoderLayer(nn.Module):
    def __init__(self, key_size, value_size, num_hiddens, num_heads, ffn_num_input, ffn_num_hiddens, ffn_num_outputs, normalized_shape, dropout, bias, **kwargs):
        super(EncoderLayer, self).__init__(**kwargs)
        self.self_attn = MultiHeadAttention(key_size, value_size, num_hiddens, num_heads, dropout, bias=bias)
        self.add_norm1 = AddNorm(normalized_shape, dropout, bias=bias)
        self.cross_attn = MultiHeadAttention(key_size, query_size, value_size, num_hiddens, num_heads, dropout, bias=bias)
        self.add_norm2 = AddNorm(normalized_shape, dropout, bias=bias)
        self.ffn = PositionWiseFFN(ffn_num_input, ffn_num_hiddens, ffn_num_outputs, **kwargs)
        self.add_norm3 = AddNorm(normalized_shape, dropout, bias=bias)
    
    def forward(self, X, enc_output, valid_lens, enc_valid_lens):
        Y = self.add_norm1(X, self.self_attn(X, X, X, valid_lens))
        Z = self.add_norm2(Y, self.cross_attn(Y, enc_output, enc_output, valid_lens, enc_valid_lens))
        return self.add_norm3(self.ffn(Z))        
    